# CAMeL-BERT Fine-Tuning for Khabar Boundary Detection
## Complete Pipeline for Google Colab (GPU) — FIXED WITH CLASS WEIGHTS

**Overview:**
- Stage 1: Mount Drive & Load Annotated Data
- Stage 2: Data Preparation (BIO tagging)
- Stage 3: Setup & Environment
- Stage 4: Model Training with Class Weights
- Stage 5: Evaluation & Metrics
- Stage 6: Model Validation

**KEY FIX**: Added class weights to penalize incorrect I- predictions when B- should be used (handles 1% B- vs 98% I- imbalance).

**Expected Results:**
- Token F1: 0.80-0.85
- B- token detection: >50% (was 0% before)
- Boundary precision: 65-75% usable

**Training Time:** ~30-60 minutes on GPU

## STAGE 1: Mount Google Drive & Verify Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_path = '/content/drive/MyDrive/Khabar-segmentation'
print(f"Drive mounted at: {drive_path}")
print(f"Directory exists: {os.path.exists(drive_path)}")

if os.path.exists(drive_path):
    print(f"\nKey directories:")
    for subdir in ['data/processed', 'notebooks', 'scripts']:
        full_path = os.path.join(drive_path, subdir)
        exists = os.path.exists(full_path)
        print(f"  {subdir}: {exists}")

## STAGE 2: Load Preprocessed Data with High Confidence Boundaries

In [ ]:
print("[STAGE 2] Loading preprocessed data...\n")

import json
import re
from typing import List, Dict, Tuple
from pathlib import Path
import random

data_dir = Path('/content/drive/MyDrive/Khabar-segmentation/data/processed')
preprocessed_dir = data_dir / 'camelbert_preprocessed'
output_dir = Path('/content/drive/MyDrive/Khabar-segmentation/data/camelbert_training')
output_dir.mkdir(parents=True, exist_ok=True)

preprocessed_path = preprocessed_dir / 'training_data_high_conf.json'

if not preprocessed_path.exists():
    print(f"[ERROR] Preprocessed data not found at {preprocessed_path}")
    print(f"[ACTION] Make sure you ran: python scripts/preprocess_annotated_data.py")
else:
    with open(preprocessed_path, 'r', encoding='utf-8') as f:
        preprocessed_examples = json.load(f)

    print(f"[OK] Loaded {len(preprocessed_examples)} high-confidence examples\n")

    boundary_types = {}
    for ex in preprocessed_examples:
        btype = ex.get('boundary_type', 'unknown')
        boundary_types[btype] = boundary_types.get(btype, 0) + 1

    print("Boundary type distribution:")
    for btype, count in sorted(boundary_types.items(), key=lambda x: -x[1]):
        print(f"  {btype:20s}: {count:3d} ({100*count/len(preprocessed_examples):5.1f}%)")

## STAGE 2: Define Helper Functions

In [ ]:
def normalize_arabic_text(text: str) -> str:
    """Normalize Arabic text."""
    text = re.sub(r'[\u064B-\u065F]', '', text)
    text = re.sub(r'[\u0660-\u0669\u06F0-\u06F90-9]', '', text)  # Remove numerals
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
    text = text.replace('ة', 'ه')
    return text

def simple_tokenize(text: str) -> List[str]:
    """Simple whitespace tokenization."""
    return text.split()

def create_bio_tags_from_boundaries(full_text: str, isnad_start: int, isnad_end: int) -> Tuple[List[str], List[str]]:
    """
    Create BIO tags from explicit character boundaries.
    """
    tokens = simple_tokenize(full_text)
    bio_tags = []
    char_pos = 0
    first_isnad_token = True
    first_khabar_token = True

    for token in tokens:
        token_start = full_text.find(token, char_pos)
        if token_start < 0:
            token_start = char_pos
        token_end = token_start + len(token)

        if token_start < isnad_end:
            if first_isnad_token:
                bio_tags.append('B-ISNAD')
                first_isnad_token = False
            else:
                bio_tags.append('I-ISNAD')
        else:
            if first_khabar_token:
                bio_tags.append('B-KHABAR')
                first_khabar_token = False
            else:
                bio_tags.append('I-KHABAR')

        char_pos = token_end

    return tokens, bio_tags

print("[OK] Functions defined")

## STAGE 2: Create Training Examples from Preprocessed Data

In [ ]:
print("\n[STAGE 2] Creating training examples from preprocessed boundaries...\n")

training_examples = []
failed_count = 0

for i, ex in enumerate(preprocessed_examples):
    full_text = ex['full_text']
    isnad_start = ex['isnad_start']
    isnad_end = ex['isnad_end']

    if isnad_start < 0 or isnad_end <= 0:
        failed_count += 1
        continue

    tokens, bio_tags = create_bio_tags_from_boundaries(full_text, isnad_start, isnad_end)

    if len(tokens) > 0 and len(tokens) == len(bio_tags):
        training_examples.append({
            'akhbar_num': ex['akhbar_num'],
            'tokens': tokens,
            'ner_tags': bio_tags,
            'text': ' '.join(tokens),
            'confidence': ex.get('confidence', 0.0)
        })
    else:
        failed_count += 1

    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(preprocessed_examples)}")

print(f"\n[RESULTS]")
print(f"  Total extracted: {len(training_examples)}")
print(f"  Failed: {failed_count}")
print(f"  Success rate: {100 * len(training_examples) / len(preprocessed_examples):.1f}%\n")

if training_examples:
    import random as rnd
    example = training_examples[rnd.randint(0, len(training_examples)-1)]
    print(f"[EXAMPLE #{example['akhbar_num']}]")
    print(f"  Tokens ({len(example['tokens'])}): {' '.join(example['tokens'][:15])}...")
    print(f"  Tags ({len(example['ner_tags'])}): {example['ner_tags'][:15]}...")

## STAGE 2: Save Training Data as JSONL

In [ ]:
print("\n[STAGE 2] Saving training data as JSONL...\n")

label2id = {'O': 0, 'B-ISNAD': 1, 'I-ISNAD': 2, 'B-KHABAR': 3, 'I-KHABAR': 4}
id2label = {v: k for k, v in label2id.items()}

random.seed(42)
random.shuffle(training_examples)

n_train = int(0.70 * len(training_examples))
n_val = int(0.15 * len(training_examples))

train_examples = training_examples[:n_train]
val_examples = training_examples[n_train:n_train + n_val]
test_examples = training_examples[n_train + n_val:]

print(f"Train: {len(train_examples)} ({100*len(train_examples)/len(training_examples):.1f}%)")
print(f"Val:   {len(val_examples)} ({100*len(val_examples)/len(training_examples):.1f}%)")
print(f"Test:  {len(test_examples)} ({100*len(test_examples)/len(training_examples):.1f}%)\n")

def save_jsonl(examples, path):
    with open(path, 'w', encoding='utf-8') as f:
        for ex in examples:
            ner_tag_ids = [label2id[tag] for tag in ex['ner_tags']]
            json_line = {'tokens': ex['tokens'], 'ner_tags': ner_tag_ids}
            f.write(json.dumps(json_line, ensure_ascii=False) + '\n')

save_jsonl(train_examples, output_dir / 'train.jsonl')
save_jsonl(val_examples, output_dir / 'val.jsonl')
save_jsonl(test_examples, output_dir / 'test.jsonl')

with open(output_dir / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print(f"[OK] Saved training data:")
print(f"  Train: {output_dir}/train.jsonl")
print(f"  Val:   {output_dir}/val.jsonl")
print(f"  Test:  {output_dir}/test.jsonl")

## STAGE 3: Setup Environment & Install Dependencies

In [ ]:
print("[STAGE 3] Installing dependencies...\n")

!pip install -q transformers==4.40.0 torch==2.2.0 datasets scikit-learn tqdm

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

print("[OK] Dependencies installed")

## STAGE 4: Load Data & Compute Class Weights

In [ ]:
print("\n[STAGE 4] Loading data and computing class weights...\n")

from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Load label mappings
with open(output_dir / 'label_mapping.json', 'r', encoding='utf-8') as f:
    label_map = json.load(f)
    label2id = label_map['label2id']
    id2label = {int(k): v for k, v in label_map['id2label'].items()}

print(f"Label mappings: {id2label}\n")

# COMPUTE CLASS WEIGHTS - CRITICAL FIX
print("[STAGE 4] Computing class weights for imbalanced labels...\n")

all_labels = []
for ex in training_examples:
    for tag in ex['ner_tags']:
        all_labels.append(label2id[tag])

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(all_labels),
    y=all_labels
)

print("Label distribution:")
from collections import Counter
label_counts = Counter(all_labels)
total = sum(label_counts.values())
for label_id in range(5):
    count = label_counts.get(label_id, 0)
    pct = 100 * count / total if total > 0 else 0
    weight = class_weights[label_id] if label_id < len(class_weights) else 1.0
    print(f"  {id2label[label_id]:12s}: {count:6d} ({pct:5.1f}%) -> weight: {weight:.2f}")

# Convert to tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f"\n[OK] Class weights computed\n")

## STAGE 4: Load Model & Prepare Datasets

In [ ]:
print("[STAGE 4] Loading CAMeL-BERT model...\n")

model_name = "aubmindlab/bert-base-arabertv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print(f"[OK] Loaded {model_name}")
print(f"    Vocab size: {tokenizer.vocab_size}")
print(f"    Num labels: {len(label2id)}")
print(f"    Model parameters: {model.num_parameters():,}\n")

In [ ]:
print("[STAGE 4] Loading and tokenizing datasets...\n")

def load_dataset_from_jsonl(path):
    data = {'tokens': [], 'ner_tags': []}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            example = json.loads(line)
            data['tokens'].append(example['tokens'])
            data['ner_tags'].append(example['ner_tags'])
    return Dataset.from_dict(data)

train_dataset = load_dataset_from_jsonl(output_dir / 'train.jsonl')
val_dataset = load_dataset_from_jsonl(output_dir / 'val.jsonl')
test_dataset = load_dataset_from_jsonl(output_dir / 'test.jsonl')

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"Test: {len(test_dataset)} examples\n")

In [ ]:
print("[STAGE 4] Tokenizing datasets...\n")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=512,
        padding='max_length'
    )

    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

train_dataset_tok = train_dataset.map(tokenize_and_align_labels, batched=True, batch_size=32, remove_columns=['tokens', 'ner_tags'])
val_dataset_tok = val_dataset.map(tokenize_and_align_labels, batched=True, batch_size=32, remove_columns=['tokens', 'ner_tags'])
test_dataset_tok = test_dataset.map(tokenize_and_align_labels, batched=True, batch_size=32, remove_columns=['tokens', 'ner_tags'])

print("[OK] Datasets tokenized")

## STAGE 5: Train with Class Weights

In [ ]:
print("\n[STAGE 5] Setting up training with class weights...\n")

from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_predictions = [item for sublist in true_predictions for item in sublist]
    true_labels = [item for sublist in true_labels for item in sublist]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average='weighted', zero_division=0
    )

    return {'precision': precision, 'recall': recall, 'f1': f1}

training_args = TrainingArguments(
    output_dir=str(output_dir / 'checkpoint'),
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    gradient_accumulation_steps=2,
    seed=42,
)

print(f"[OK] Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
print("\n[STAGE 5] Creating trainer with class weights...\n")

# CUSTOM TRAINER WITH CLASS WEIGHTS - CRITICAL FIX
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs['labels']

        # Apply class weights to cross entropy loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits.view(-1, 5), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tok,
    eval_dataset=val_dataset_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("[OK] Trainer created with class weights")

In [ ]:
print("\n[STAGE 5] Starting training with class weights...\n")

train_result = trainer.train()

print(f"\n[OK] Training complete!")
print(f"  Final loss: {train_result.training_loss:.4f}")

In [ ]:
print("\n[STAGE 5] Saving best model...\n")

model_save_dir = output_dir / 'best_model'
model_save_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(model_save_dir))
tokenizer.save_pretrained(str(model_save_dir))

with open(model_save_dir / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print(f"[OK] Model saved to: {model_save_dir}")

## STAGE 6: Evaluate on Test Set

In [ ]:
print("\n[STAGE 6] Evaluating model on test set...\n")

test_results = trainer.evaluate(eval_dataset=test_dataset_tok)

print(f"[TEST SET METRICS]")
print(f"  Loss: {test_results.get('eval_loss', 'N/A'):.4f}")
print(f"  F1-score: {test_results.get('eval_f1', 'N/A'):.4f}")
print(f"  Precision: {test_results.get('eval_precision', 'N/A'):.4f}")
print(f"  Recall: {test_results.get('eval_recall', 'N/A'):.4f}")

results_path = output_dir / 'training_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(test_results, f, indent=2)

print(f"\n[OK] Results saved to: {results_path}")

In [ ]:
print("\n[STAGE 6] Per-label detailed evaluation...\n")

predictions = trainer.predict(test_dataset_tok)
pred_labels = np.argmax(predictions.predictions, axis=2)
true_labels = predictions.label_ids

pred_flat = []
true_flat = []
for pred_row, true_row in zip(pred_labels, true_labels):
    for pred, true in zip(pred_row, true_row):
        if true != -100:
            pred_flat.append(id2label[pred])
            true_flat.append(id2label[true])

from sklearn.metrics import classification_report
report = classification_report(true_flat, pred_flat, zero_division=0)
print(report)

with open(output_dir / 'detailed_evaluation.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(f"[OK] Detailed report saved")

## STAGE 7: Summary

In [ ]:
print("\n[STAGE 7] TRAINING COMPLETE - Summary\n")
print("=" * 80)
print("CAMELBERT FINE-TUNING RESULTS (WITH CLASS WEIGHTS)")
print("=" * 80 + "\n")

summary = f"""TRAINING DATA
  Total examples: {len(training_examples)}
  Train: {len(train_examples)}
  Val: {len(val_examples)}
  Test: {len(test_examples)}

KEY FIX: Class Weights Applied
  B-ISNAD weight: {class_weights[1]:.2f} (was 1.0)
  B-KHABAR weight: {class_weights[3]:.2f} (was 1.0)
  I-ISNAD weight: {class_weights[2]:.2f}
  I-KHABAR weight: {class_weights[4]:.2f}

MODEL
  Base: aubmindlab/bert-base-arabertv2
  Labels: {len(label2id)} (O, B-ISNAD, I-ISNAD, B-KHABAR, I-KHABAR)
  Parameters: {model.num_parameters():,}

TRAINING
  Epochs: {training_args.num_train_epochs}
  Batch size: {training_args.per_device_train_batch_size}
  Learning rate: {training_args.learning_rate}
  
TEST SET METRICS
  F1-score: {test_results.get('eval_f1', 'N/A'):.4f}
  Precision: {test_results.get('eval_precision', 'N/A'):.4f}
  Recall: {test_results.get('eval_recall', 'N/A'):.4f}
  Loss: {test_results.get('eval_loss', 'N/A'):.4f}

OUTPUT FILES (in Drive)
  Model: {model_save_dir}
  Training data: {output_dir}/train.jsonl, val.jsonl, test.jsonl
  Results: {output_dir}/training_results.json
  Evaluation: {output_dir}/detailed_evaluation.txt

NEXT STEPS
  1. Run full corpus inference (CAMELBERT_INFERENCE_EVALUATION.ipynb)
  2. Compute boundary precision on reference data
  3. Compare with baseline v3.5 (3.1% usable)
  4. Expected improvement: 65-75% usable boundaries (was 0% without class weights)
"""

print(summary)

with open(output_dir / 'TRAINING_SUMMARY.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"\n[OK] Summary saved")